In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
%cd drive/MyDrive/

In [ ]:
!rm -rf FPL_forecast
!git clone https://github.com/bragehs/FPL_forecast.git

In [ ]:
%cd FPL_forecast/predictor/

In [3]:
file_path = '/content/drive/MyDrive/colab_fpl'
file_path

'/content/drive/MyDrive/colab_fpl'

In [13]:
import os
import torch
from training import train_model, hyperparameter_tuning
from model import AdvancedLSTM
import json

In [14]:
file_path = os.getcwd() + "/processed_data"
file_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [15]:
X_train = torch.load(file_path + "/X_train.pt", weights_only=True)
y_train = torch.load(file_path + "/y_train.pt", weights_only=True)
X_val = torch.load(file_path + "/X_val.pt", weights_only=True)
y_val = torch.load(file_path + "/y_val.pt", weights_only=True)

print(f"Train sequences: {X_train.shape}, Targets: {y_train.shape}")

Train sequences: torch.Size([192296, 5, 31]), Targets: torch.Size([192296, 1])


In [16]:
player_ids_train = torch.load(file_path + "/train_player_ids.pt", weights_only=True)
player_ids_val = torch.load(file_path + "/val_player_ids.pt", weights_only=True)
pos_ids_train = torch.load(file_path + "/pos_train.pt", weights_only=True)
pos_ids_val = torch.load(file_path + "/pos_val.pt", weights_only=True)
fix_diff_train = torch.load(file_path + "/fixdiff_train.pt", weights_only=True)
fix_diff_val = torch.load(file_path + "/fixdiff_val.pt", weights_only=True)

In [17]:
print(f"Player IDs Train: {player_ids_train.shape}, Val: {player_ids_val.shape}")
print(f"Position IDs Train: {pos_ids_train.shape}, Val: {pos_ids_val.shape}")
print(f"Fixture Difficulty Train: {fix_diff_train.shape}, Val: {fix_diff_val.shape}")

Player IDs Train: torch.Size([192296]), Val: torch.Size([27283])
Position IDs Train: torch.Size([192296, 5]), Val: torch.Size([27283, 5])
Fixture Difficulty Train: torch.Size([192296, 5]), Val: torch.Size([27283, 5])


In [19]:
print(fix_diff_train.unique())
print(pos_ids_train.unique())

tensor([0, 2, 3, 4, 5])
tensor([0, 1, 2, 3, 4])


In [10]:
with open(f"{file_path}/vocab/player_name_to_idx.json") as f:
    name_to_idx = json.load(f)
with open(f"{file_path}/vocab/unk_id.txt") as f:
    unk_id = int(f.read())

In [23]:
# Hyperparameter tuning
best_params = hyperparameter_tuning(X_train, y_train, X_val, y_val,
                                    player_ids_train=player_ids_train, player_ids_val=player_ids_val,
                                    player_vocab_size=len(name_to_idx), player_embed_dim=32,unknown_player_index=unk_id,
                                    pos_ids_train=pos_ids_train, pos_ids_val=pos_ids_val,
                                    position_vocab_size= len(pos_ids_train.unique())+1, position_embed_dim=16,
                                    fixdiff_ids_train=fix_diff_train, fixdiff_ids_val=fix_diff_val,
                                    fixture_diff_vocab_size=len(fix_diff_train.unique())+1, fixture_diff_embed_dim=16,
                                    epochs=1, n_trials=1, transform=False, num_workers=4,
                                    )

# Full training with best hyperparameters
print("\nTraining final model with best hyperparameters...")
adv_model = AdvancedLSTM(input_dim=X_train.shape[-1], hidden_dim=best_params['hidden_dim'],
                            output_dim=1, num_layers=best_params['num_layers'],
                            dropout=best_params['dropout'], num_fc_layers=best_params['num_fc_layers'],
                            player_vocab_size=len(name_to_idx), player_embed_dim=32,unknown_player_index=unk_id,
                            position_vocab_size=len(pos_ids_train.unique())+1, position_embed_dim=16,
                            fixture_diff_vocab_size=len(fix_diff_train.unique())+1, fixture_diff_embed_dim=16,
                            )


train_model(
    adv_model,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    player_ids_train=player_ids_train,
    player_ids_val=player_ids_val,
    pos_ids_train=pos_ids_train,
    pos_ids_val=pos_ids_val,
    fixdiff_ids_train=fix_diff_train,
    fixdiff_ids_val=fix_diff_val,
    learning_rate=best_params['learning_rate'],
    weight_decay=best_params['weight_decay'],
    batch_size=best_params['batch_size'],
    epochs=100,
    verbose=1,
    transform=False,
    num_workers=4,
        )

Running random search with 1 trials...

Trial 1/1
Params: {'learning_rate': 0.001, 'hidden_dim': 192, 'weight_decay': 1e-06, 'num_layers': 2, 'dropout': 0.3, 'num_fc_layers': 2, 'batch_size': 64}


New best RMSE: 1.7142

Top 5 hyperparameter combinations:
1. RMSE: 1.7142, Params: {'learning_rate': 0.001, 'hidden_dim': 192, 'weight_decay': 1e-06, 'num_layers': 2, 'dropout': 0.3, 'num_fc_layers': 2, 'batch_size': 64, 'rmse': np.float64(1.7141569107283898)}

Best hyperparameters: {'learning_rate': 0.001, 'hidden_dim': 192, 'weight_decay': 1e-06, 'num_layers': 2, 'dropout': 0.3, 'num_fc_layers': 2, 'batch_size': 64}
Best RMSE: 1.7142

Training final model with best hyperparameters...


libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x12910dc60>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1568, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 